# 0.1 · The real credit data

*0. General · notebook 0.1 of the story.* ← (start of the story) · [0.2 · the PD prior](<0.2_prior_visualisation_pd.ipynb>) →

**The measurement instrument, not the method.** These 21 real credit datasets — 14 PD and 7 LGD — are
what every model in this project is *scored* on; they are neither training data nor priors. Before a
prior is designed, this notebook fixes what it has to reproduce: how rare default is, where the loss
mass sits, what the tables look like, and whether a number built on them can be trusted.

Everything reads the processed parquet cache through `src.data.pipeline`, so this notebook sees exactly
the tables the evaluation sees — not a separate read of the raw files that might disagree. Anything not
yet processed is processed on first access, which takes a few minutes once.

**How to read it.** Part A is the targets — the thing this project is about, and what the credit
mechanisms in 0.2 and 0.3 are built to match. Part B is the features a model reads them from. Part C
checks the evaluation itself. There is no colour key here: the prior colours mean nothing for real data.

## Grounding in the literature

Every gap this notebook documents is one `tfm-library` names as unmet by the default TFM prior.
**Imbalance:** real PD base rates sit far below balance, and below roughly 10 % a default-threshold
classifier collapses to the majority class (`papers/2026/05_Tanna_DataPresentation` §5.1); O'Prior
evaluates classification only and lightly filters severely imbalanced tasks
(`papers/2026/05_Bouadi_ShapingThePrior`), so a *targeted* low base rate is untried. **Bounded targets:**
the same paper never exercises its regression branch, leaving LGD's [0, 1]-with-atoms shape open.
**Missingness:** TabICLv2 mean-imputes at inference (`repositories/TabICL.txt` `TransformToNumerical`),
discarding the signal a thin credit file carries, and the GBDT-over-TFM gap grows with the
missing-value fraction (ρ = +0.29) and with high-cardinality categoricals (ρ = +0.47;
`papers/2026/06_Purucker_BeyondIID` Table E.3). Pin `e5ce016`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, pathlib
ROOT = pathlib.Path.cwd()
# Walk up to the repository root — the notebook may be opened from its chapter folder
# (notebooks/1. Experiment 1/), from notebooks/, or from the root — then work FROM the root,
# so relative paths (config/...) resolve exactly as under `python -m src.utils.run_notebooks`.
while not (ROOT / "src" / "visualize").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src.visualize import data_plots, figures, style, summaries, literature

style.apply()   # ONE shared style: identical colours in every notebook
pd.set_option("display.width", 220, "display.max_columns", 60, "display.max_rows", 60)

# Clears THIS notebook's figure folder — and no other — BEFORE anything is drawn, then
# saves every figure as a PDF sized for A4. Identical in Jupyter and under the runner.
FIGS = figures.FigureSaver("0.1_data_exploration")

## Loading the datasets

A dataset that cannot be read is **skipped with a message** rather than killing the notebook — one bad
raw file should not stop you looking at the other twenty.

In [ ]:
lgd = data_plots.load_all("lgd")
pd_ = data_plots.load_all("pd")
both = {"lgd": lgd, "pd": pd_}
print(f"\nloaded {len(lgd)} LGD and {len(pd_)} PD datasets")

## A · What are we predicting?

The two targets, as they really are. Everything the credit prior adds — a controlled base rate, boundary atoms — is aimed at what these figures show.

### A1 · PD — how rare is default?

The table restates each base rate as odds, which is easier to feel: 1 : 99 is one default per hundred
loans. Real books run from about 7 % (GMSC; Home Credit at 8 %, as `papers/2026/05_Tanna_DataPresentation`
§4.1 also reports) up to 40 % — all well below the balance TabICL's unmodified prior sits near. That gap
is what the PD arm addresses, and it is why the results notebooks never report accuracy alone.

**How to read it.** The dashed line is a 50/50 split — roughly what TabICL's unmodified prior produces. Every real dataset sits well left of it, which is the gap our prior targets. The rate axis is logarithmic because the rates span two orders of magnitude; on a
linear axis every dataset below 5 % collapses into the same sliver.

In [ ]:
data_plots.summary_table(pd_, 'pd')

In [ ]:
FIGS.save(
    data_plots.plot_pd_base_rates(pd_),
    "pd_base_rates",
    caption=(
        "Default rate per PD dataset, ordered by rate, on a logarithmic horizontal axis. The "
        "dashed vertical line marks a 50% rate. Percentages give the rate per dataset."
    ),
);


### A2 · LGD — where does the loss mass sit?

The motivating figure of the project. Read **boundary mass** first — the share of rows exactly at the
minimum or maximum — and whether every target lies inside [0, 1]. Where boundary mass is large, a model
that predicts only smooth interior values cannot be calibrated: it can get the average right while
being wrong about every individual loan.

**How to read it.** Dashed red lines mark values that are *exactly* the minimum or maximum. In a 40-bin histogram an exact atom at 0 and a cluster at 0.02 look identical, and only the first one is the problem.

In [ ]:
data_plots.summary_table(lgd, 'lgd')

In [ ]:
FIGS.save(
    data_plots.plot_lgd_targets(lgd),
    "lgd_targets",
    caption=(
        "Histograms of the Loss Given Default target for each of the seven LGD datasets, ordered "
        "by sample size. Forty bins per panel. Dashed vertical lines mark the minimum and maximum "
        "observed values where more than 1% of observations fall exactly on them. Panel subtitles "
        "give the combined share of observations at the two boundaries."
    ),
);


### A3 · LGD — which books does it matter for?

The same boundary mass, ranked, because that is the practical question. Two things to take away: the
spread is **wide**, so the prior needs a *range* of boundary masses rather than one value; and mass at 0
(full recovery) and at 1 (total loss) are **not symmetric**, so the prior samples them separately.

**How to read it.** The split shows *which* boundary the mass sits on. Two books with the same total behave very differently if one recovers in full and the other is a total loss.

In [ ]:
FIGS.save(
    data_plots.plot_boundary_mass_ranking(lgd),
    "boundary_mass_ranking",
    caption=(
        "Share of observations lying exactly at a boundary of the observed target range, per LGD "
        "dataset, ordered by total. Bars are split into mass at the minimum (blue) and at the "
        "maximum (orange). Percentages give the total per dataset."
    ),
);


## B · What does a model read the target from?

The tables themselves. They set the ranges the prior samples over — table size, feature types, missingness and dependence.

### B1 · Table size

If the prior generated 500-column tables and every real dataset had 20, the extra capacity would be
wasted; if it generated 200-row tables and real ones have a million, the model would never learn to use
a long context.

**How to read it.** Both axes are logarithmic. Position matters more than the exact values: real credit tables are wide and long-tailed in size.

In [ ]:
FIGS.save(
    data_plots.plot_shapes(both),
    "shapes",
    caption=(
        "Number of rows against number of features for all 21 evaluation datasets, both axes "
        "logarithmic. Colour denotes task; each point is labelled with its dataset name."
    ),
);


### B2 · Feature types

The mix of categorical and numerical features per dataset. Credit tables mix numerical measurements with
categoricals — product codes, regions, employment classes — and high-cardinality categoricals are exactly
where the GBDT-over-TFM gap grows (ρ = +0.47, `papers/2026/06_Purucker_BeyondIID` Table E.3).

In [ ]:
FIGS.save(
    data_plots.plot_type_mix(both),
    "type_mix",
    caption=(
        "Share of columns that are categorical, per dataset, ordered by share. Colour denotes "
        "task."
    ),
);


### B3 · Missingness

**How to read it.** Measured *after* preprocessing, so this is what a model actually receives, not what the raw file contained.

Many datasets have **zero** missing values: most were imputed before we received them, so real
missingness is understated here. The prior still injects missingness as an explicit mechanism (0.2 B5,
0.3 B4) — a model that has never seen a missing value handles one badly, and TabICLv2 mean-imputes it
away at inference (`repositories/TabICL.txt` `TransformToNumerical`). These numbers measure the upstream
pipeline, not the domain, so the prior is not tuned to them.

In [ ]:
FIGS.save(
    data_plots.plot_missingness(both),
    "missingness",
    caption=(
        "Share of cells that are missing, per dataset, ordered by share, measured after "
        "preprocessing. Colour denotes task."
    ),
);


### B4 · Feature dependence

Real credit data comes in **blocks** of strongly correlated columns — several measures of one balance,
several vintages of one delinquency count. The prior builds its features through random DAGs precisely
so those blocks appear; if these heatmaps were diagonal, that design choice would be wrong. Every dataset
is shown, paginated, never only the largest.

**How to read it.** Blocks of red off the diagonal are groups of features measuring the same thing — several views of one balance. A prior generating independent columns would look nothing like this, which is the point O'Prior makes.

In [ ]:
# Every dataset, six per page: at A4 width more than six panels stops being readable,
# and showing only the six largest would describe "real credit data" from half of it.
for _p in range(1, data_plots.correlation_pages(lgd) + 1):
    FIGS.save(
        data_plots.plot_feature_correlations(lgd, page=_p),
        f"feature_correlations_lgd_p{_p}",
        caption=(
        "Pearson correlation matrices between features, one panel per dataset, computed on "
        "the first 5,000 rows with constant columns removed. Colour scale spans -1 to 1. "
        "Panel headings give the dataset and the number of columns retained."
            f" LGD datasets, page {_p} of {data_plots.correlation_pages(lgd)}."
        ),
    )


In [ ]:
for _p in range(1, data_plots.correlation_pages(pd_) + 1):
    FIGS.save(
        data_plots.plot_feature_correlations(pd_, page=_p),
        f"feature_correlations_pd_p{_p}",
        caption=(
        "Pearson correlation matrices between features, one panel per dataset, computed on "
        "the first 5,000 rows with constant columns removed. Colour scale spans -1 to 1. "
        "Panel headings give the dataset and the number of columns retained."
            f" PD datasets, page {_p} of {data_plots.correlation_pages(pd_)}."
        ),
    )


## C · Can the evaluation be trusted?

A benchmark is only as good as its datasets. One check that has already caught something.

### C1 · Leakage screen

Every column's absolute correlation with its target. This exists because `lgd_lendingclub` gives R²
around 0.71–0.76 — far above anything reported for LGD, which usually means a column encodes the answer.

A high correlation is a **pointer, not proof**: one strong predictor can be legitimate. And the screen
looks at one feature at a time, so it cannot see leakage spread across several columns.

In [ ]:
leak_lgd = data_plots.leakage_check(lgd, "lgd")
leak_lgd.head(12)

In [ ]:
leak_pd = data_plots.leakage_check(pd_, "pd")
leak_pd.head(12)

In [ ]:
flagged = pd.concat([leak_lgd, leak_pd])
flagged[flagged["suspicious"]]

## Summary

The datasets in text, PD first and then LGD — each task's tables and its target — followed by the
leakage screen. Printed last so `output/All_Results.md` carries the numbers, then the `tfm-library`
sources (pin `e5ce016`) and the figure inventory.

In [ ]:
print(summaries.data_summary({"pd": pd_, "lgd": lgd}, leakage=pd.concat([leak_pd, leak_lgd])))
print()
print(literature.references_md(["tanna_paradox", "hc_base_rate", "tabicl_impute", "purucker_highcard", "purucker_missing", "oprior_scope"]))
print()
print(FIGS.summary())